# Week 9 · Day 4 — CNN Architecture, Done Properly (Malaria Detection)

Yesterday you built your first CNN and watched it beat the ANN on faces. Today we turn “a CNN” into “a **well-built** CNN” — and we do it on something real: **detecting malaria from blood-cell images.**

Given a microscope image of a single red blood cell, the network decides: **infected** (parasitized) or **healthy** (uninfected). This is a genuine medical-imaging task, and by the end of today a from-scratch CNN will do it at **~95% accuracy**. You will have built a malaria detector.

Along the way we’ll finally cover the architectural pieces we’ve been using without explaining: **pooling, stride, and padding**, and the standard CNN blueprint that ties them together.

**Today’s plan:**
1. Load the malaria cell images from folders with `os`.
2. **The building blocks** — pooling, stride, padding, explained.
3. **The full CNN blueprint** — and tracking the data shape through it.
4. Train a proper CNN to ~95% and evaluate it.
5. **Overfitting & how to fight it** — dropout and data augmentation.

> A colorful, real dataset and a satisfying result. Let’s build something that works.

> **Running on Kaggle GPU:** enable it under **Settings > Accelerator > GPU**, then add the malaria dataset via **Add Input**. The code detects the GPU automatically and moves the model and every batch to it.

In [ ]:
# install if needed (Kaggle already has torch + pillow + sklearn)
# !pip install torch pillow scikit-learn

import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

# --- pick the GPU if one is available (Kaggle: turn on GPU in Settings > Accelerator) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch version:", torch.__version__)
print("using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — on Kaggle, enable it via Settings > Accelerator > GPU.")

---
## 1. Load the malaria cell images with a custom `Dataset`

The dataset is two folders, one per class:
```
cell_images/
  Parasitized/   ...png   (infected cells)
  Uninfected/    ...png   (healthy cells)
```
There’s no pre-made train/test split, so we’ll make our own.

**The proper PyTorch way to handle images: a custom `Dataset`.** Instead of loading all ~27,000 images into memory up front, we:
1. Walk the folders with `os` and collect just the **file paths and labels** (cheap — no images loaded yet).
2. Write a `Dataset` class that opens and resizes **one image at a time**, only when it’s actually needed for a batch.

This is how real image pipelines work — it scales to datasets far too big to fit in memory, because only the current batch is ever loaded.

In [ ]:
# ---- point this at the folder containing the two class subfolders ----
# On Kaggle, add the dataset via 'Add Input' and it appears under /kaggle/input/.
# The NIH malaria dataset commonly extracts to a 'cell_images' folder, e.g.:
#   /kaggle/input/cell-images-for-detecting-malaria/cell_images/cell_images
# Set DATA_DIR to whichever folder directly contains Parasitized/ and Uninfected/.
DATA_DIR = "/kaggle/input/cell-images-for-detecting-malaria/cell_images/cell_images"

VALID_EXT = (".png", ".jpg", ".jpeg")

class_names = sorted([n for n in os.listdir(DATA_DIR)
                      if os.path.isdir(os.path.join(DATA_DIR, n))])
print("classes found:", class_names)   # ['Parasitized', 'Uninfected']

# collect ONLY paths + labels (no images loaded yet — this is instant)
all_paths, all_labels = [], []
for label_index, class_name in enumerate(class_names):
    folder = os.path.join(DATA_DIR, class_name)
    for filename in os.listdir(folder):
        if filename.lower().endswith(VALID_EXT):
            all_paths.append(os.path.join(folder, filename))
            all_labels.append(label_index)

print(f"found {len(all_paths)} image paths across {len(class_names)} classes")
for label_index, class_name in enumerate(class_names):
    print(f"  {class_name:14s}: {all_labels.count(label_index)}")

### The custom `Dataset` class
A PyTorch `Dataset` needs just three methods:
- **`__init__`** — store the list of paths and labels (and any transform).
- **`__len__`** — how many items there are.
- **`__getitem__(i)`** — load, resize, and return item *i* as a `(tensor, label)` pair. **This is where an image is actually read from disk** — one at a time, on demand.

Malaria cell images come in different sizes, so we resize every image to a fixed **64×64 color** inside `__getitem__`.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

IMG_SIZE = 64

class MalariaDataset(Dataset):
    """Stores only file paths; loads each image lazily in __getitem__."""
    def __init__(self, paths, labels, img_size=64):
        self.paths = paths
        self.labels = labels
        self.img_size = img_size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        # open ONE image from disk, resize, scale to 0-1, to (channels, H, W)
        img = Image.open(self.paths[i]).convert("RGB").resize((self.img_size, self.img_size))
        arr = np.asarray(img, dtype="float32") / 255.0        # (H, W, 3)
        tensor = torch.from_numpy(arr).permute(2, 0, 1)        # -> (3, H, W)
        return tensor, self.labels[i]

# split the PATHS (not the images) into train/test, keeping class balance
train_paths, test_paths, train_labels, test_labels = train_test_split(
    all_paths, all_labels, test_size=0.2, random_state=42, stratify=all_labels)

train_ds = MalariaDataset(train_paths, train_labels, IMG_SIZE)
test_ds  = MalariaDataset(test_paths,  test_labels,  IMG_SIZE)

print("train items:", len(train_ds), "  test items:", len(test_ds))
# peek at one item — this triggers a single image load
sample_x, sample_y = train_ds[0]
print("one item ->", tuple(sample_x.shape), "  label:", sample_y, "(", class_names[sample_y], ")")

### Look at the data first
Infected cells usually show a dark purple/pink spot (the parasite); healthy cells are clean. We pull a few items straight from the dataset (each access loads one image on demand).

In [ ]:
# grab a few examples of each class from the training dataset
fig, axes = plt.subplots(2, 6, figsize=(13, 4.5))
for row, label_index in enumerate(range(len(class_names))):
    # find dataset indices whose label is this class
    idxs = [i for i, lab in enumerate(train_labels) if lab == label_index][:6]
    for col, i in enumerate(idxs):
        img_t, lab = train_ds[i]                      # loads one image
        axes[row, col].imshow(img_t.permute(1, 2, 0).numpy())   # (3,H,W)->(H,W,3)
        axes[row, col].set_title(class_names[lab], fontsize=9)
        axes[row, col].axis("off")
plt.suptitle("Top row: one class   |   Bottom row: the other  (spot the parasite)")
plt.tight_layout()
plt.show()

### Wrap the datasets in `DataLoader`s
The `DataLoader` pulls items from our `Dataset` in shuffled **batches** — calling `__getitem__` behind the scenes, so only one batch of images is in memory at a time. This is the whole payoff of the path-based approach: it scales to any dataset size.

We also make a small helper to measure accuracy by streaming through a loader (since the data no longer lives in one big tensor we can evaluate in one shot).

In [ ]:
BATCH_SIZE = 64
# num_workers parallelises image loading; pin_memory speeds host->GPU copies
loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=(device.type=="cuda"))
train_loader = DataLoader(train_ds, shuffle=True,  **loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kwargs)

n_classes = len(class_names)
print("batches per epoch (train):", len(train_loader))
print("CNN input shape per batch: (batch, 3 channels, 64, 64)")

@torch.no_grad()
def accuracy(model, loader):
    """Stream through a loader and return accuracy (batches moved to device)."""
    model.eval()
    correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(1)
        correct += (pred == yb).sum().item()
        total += len(yb)
    return correct / total

---
## 2. The building blocks

We’ve used pooling already without really explaining it. Let’s properly meet the three knobs that shape a CNN: **pooling, stride, and padding.** These control the *size* of the data as it flows through — and tracking that size is the key skill for building CNNs.

### Pooling — shrink while keeping the strongest signal
**Max pooling** slides a small window (usually 2×2) over the feature map and keeps only the **largest** value in each window. A 2×2 max-pool **halves** the height and width. Why do it?
- **Efficiency** — fewer numbers to process in later layers.
- **Robustness** — keeping the strongest response makes the network less sensitive to a feature shifting by a pixel or two.

Let’s see it on a tiny grid.

In [ ]:
small = torch.tensor([[1., 3., 2., 4.],
                      [5., 6., 1., 2.],
                      [7., 2., 9., 1.],
                      [3., 4., 2., 8.]]).reshape(1, 1, 4, 4)

pool = nn.MaxPool2d(2)
result = pool(small)

print("before (4x4):\n", small[0, 0].numpy())
print("\nafter 2x2 max-pool (2x2):\n", result[0, 0].numpy())
print("\nEach output = the biggest number in its 2x2 window. Size halved: 4x4 -> 2x2.")

### Stride — how far the filter jumps
**Stride** is the step size as a filter (or pool) slides across the image. Stride 1 moves one pixel at a time (output nearly the same size). Stride 2 jumps two pixels, which **halves** the output — another way to shrink, sometimes used instead of pooling.

### Padding — a border so edges are treated fairly
Without padding, a 3×3 filter can’t centre on the edge pixels, so the output shrinks a little and edge information is under-used. **Padding** adds a border of zeros so the output stays the same size and edge pixels get full attention. `padding=1` with a 3×3 filter keeps the size unchanged — which is why we use it.

Let’s watch these three change the output size.

In [ ]:
x = torch.randn(1, 3, 64, 64)   # one fake color image, 64x64

# a conv that KEEPS size: 3x3 filter, padding=1, stride=1
conv_same = nn.Conv2d(3, 8, kernel_size=3, padding=1, stride=1)
print("conv 3x3, padding=1, stride=1:", tuple(conv_same(x).shape[2:]), "(size kept)")

# a conv that HALVES size: stride=2
conv_stride = nn.Conv2d(3, 8, kernel_size=3, padding=1, stride=2)
print("conv 3x3, padding=1, stride=2:", tuple(conv_stride(x).shape[2:]), "(halved by stride)")

# max-pool halves size too
print("after 2x2 max-pool:           ", tuple(nn.MaxPool2d(2)(x).shape[2:]), "(halved by pooling)")

# no padding shrinks a little
conv_nopad = nn.Conv2d(3, 8, kernel_size=3, padding=0)
print("conv 3x3, padding=0:          ", tuple(conv_nopad(x).shape[2:]), "(shrank: no padding)")

**The takeaway:** `padding=1` with a 3×3 conv keeps the size; **pooling** or **stride 2** halves it. In our network, convs will *keep* the size and pooling will *shrink* it — the standard, predictable pattern. Keeping track of these sizes is what stops the dreaded shape-mismatch error.

---
## 3. The full CNN blueprint

Almost every classic CNN follows one pattern:

> **[ Conv → ReLU → Pool ] × N  →  Flatten  →  Dense → output**

A stack of conv-pool blocks (the **feature extractor**), then a couple of dense layers (the **classifier**). Each block’s conv *keeps* the size (padding=1) and its pool *halves* it, so the picture gets smaller and deeper as it flows through.

Let’s track the shape through our network so the flatten size is no mystery:

| Stage | Shape (channels × H × W) |
|---|---|
| input | 3 × 64 × 64 |
| block 1: conv→pool | 16 × 32 × 32 |
| block 2: conv→pool | 32 × 16 × 16 |
| block 3: conv→pool | 64 × 8 × 8 |
| flatten | 64 × 8 × 8 = **4096** |

That **4096** is what the first dense layer must expect. Get the shape-tracking right and the network just works.

In [ ]:
class MalariaCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            # block 1: 3 -> 16 channels, then halve 64 -> 32
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            # block 2: 16 -> 32, halve 32 -> 16
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            # block 3: 32 -> 64, halve 16 -> 8
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                 # 64 * 8 * 8 = 4096
            nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(42)
model = MalariaCNN(n_classes).to(device)   # move model to GPU
print(model)

### Quick shape check (provided ✅)
Before training, confirm the data flows through with the right output shape — this catches the flatten-size bug immediately.

In [ ]:
with torch.no_grad():
    dummy = torch.randn(4, 3, 64, 64).to(device)
    out = model(dummy)
print("output shape:", tuple(out.shape), " (should be (4,", n_classes, "))")
assert out.shape == (4, n_classes), "shape mismatch — check the flatten size (64*8*8)"
print("✅ shapes line up — ready to train.")

---
## 4. Train the CNN

Same four-move training loop you’ve used all week — forward → loss → backward → update. We track **both train and test accuracy** each epoch so we can watch for overfitting (which we’ll address in Part 5).

In [ ]:
def train_model(model, loader, epochs=12, lr=0.001, log=True):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_accs, test_accs = [], []
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)      # move batch to GPU
            preds = model(xb)
            loss = loss_fn(preds, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        tr = accuracy(model, train_loader)
        te = accuracy(model, test_loader)
        train_accs.append(tr); test_accs.append(te)
        if log:
            print(f"epoch {epoch+1:2d}  train acc {tr:.2%}   test acc {te:.2%}")
    return train_accs, test_accs

train_accs, test_accs = train_model(model, train_loader, epochs=12)

In [ ]:
plt.plot([a*100 for a in train_accs], marker="o", label="train", color="purple")
plt.plot([a*100 for a in test_accs], marker="o", label="test", color="green")
plt.xlabel("epoch"); plt.ylabel("accuracy (%)")
plt.title("Malaria CNN — training vs test accuracy")
plt.legend(); plt.grid(alpha=0.3)
plt.show()
print(f"final test accuracy: {test_accs[-1]:.2%}")

### Evaluate the detector

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# collect predictions + truths by streaming the test loader
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        all_pred.extend(model(xb).argmax(1).cpu().tolist())
        all_true.extend(yb.tolist())
test_acc = np.mean(np.array(all_pred) == np.array(all_true))

cm = confusion_matrix(all_true, all_pred)
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False)
plt.title(f"Malaria detector — {test_acc:.1%} accuracy")
plt.tight_layout()
plt.show()

In a medical setting the confusion matrix matters more than the accuracy number: a **false negative** (calling an infected cell healthy) is far more dangerous than a false positive. Reading the off-diagonal cells this way is exactly how you’d judge a real diagnostic tool. Worth discussing which mistake the model makes more often.

In [ ]:
# see the detector at work — green = correct, red = wrong
model.eval()
plt.figure(figsize=(13, 5))
with torch.no_grad():
    for i in range(10):
        img_t, true_lab = test_ds[i]                       # loads one image (on CPU)
        pred = model(img_t.unsqueeze(0).to(device)).argmax(1).item()
        plt.subplot(2, 5, i+1)
        plt.imshow(img_t.permute(1, 2, 0).numpy())         # plot from the CPU tensor
        plt.title(class_names[pred], color="green" if pred == true_lab else "red", fontsize=9)
        plt.axis("off")
plt.suptitle("The malaria detector's predictions")
plt.tight_layout()
plt.show()

---
## 5. Overfitting & how to fight it

Look again at the train-vs-test curve. If **train accuracy climbs above test accuracy** and the gap grows, the network is starting to **memorise** the training cells instead of learning general features — that’s **overfitting**. On a medical tool, that’s dangerous: it looks great in training and fails on new patients.

Two standard defenses:

### Dropout
**`nn.Dropout(p)`** randomly switches off a fraction `p` of neurons during each training step. The network can’t lean on any single neuron, so it learns more robust, redundant features. It’s only active during training. Let’s add it to the classifier and compare.

In [ ]:
class MalariaCNN_Dropout(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
            nn.Dropout(0.5),                # <-- new: drop 50% of neurons in training
            nn.Linear(128, n_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(42)
model_d = MalariaCNN_Dropout(n_classes).to(device)   # move to GPU
train_d, test_d = train_model(model_d, train_loader, epochs=12, log=False)

print(f"WITHOUT dropout:  train {train_accs[-1]:.2%}  test {test_accs[-1]:.2%}  gap {(train_accs[-1]-test_accs[-1])*100:+.1f} pts")
print(f"WITH dropout:     train {train_d[-1]:.2%}  test {test_d[-1]:.2%}  gap {(train_d[-1]-test_d[-1])*100:+.1f} pts")
print("\nDropout usually shrinks the train-test gap: less memorising, better generalisation.")

### Data augmentation (the idea)
The other big defense: **make more training data for free** by randomly transforming images each epoch — small rotations, flips, brightness shifts. The network sees a slightly different version of each cell every time, so it can’t memorise exact pixels and learns what a parasite *really* looks like.

We won’t rebuild the whole pipeline for it today, but here’s the one-liner idea — a horizontal flip is a valid transform for cells (a flipped cell is still the same cell):

In [ ]:
# demonstrate augmentation on one image: original vs flipped vs rotated
from PIL import Image as PILImage
img_t, _ = train_ds[0]                                  # one image tensor (3,H,W)
sample = (img_t.permute(1, 2, 0).numpy() * 255).astype("uint8")
pil = PILImage.fromarray(sample)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
axes[0].imshow(pil);                                      axes[0].set_title("original")
axes[1].imshow(pil.transpose(PILImage.FLIP_LEFT_RIGHT)); axes[1].set_title("flipped")
axes[2].imshow(pil.rotate(25));                          axes[2].set_title("rotated 25°")
for a in axes: a.axis("off")
plt.suptitle("Data augmentation: same cell, new training examples")
plt.tight_layout()
plt.show()
print("Each is still the same cell/class — free extra training data that fights overfitting.")

### Your turn (practice) ✍️
Pick at least two:
1. **Add a 4th conv block** (`Conv2d(64, 128) → ReLU → MaxPool2d(2)`). Update the flatten size! (8→4, so `128*4*4`.) Does accuracy improve?
2. **Try a different dropout rate** (0.2, 0.7) in the dropout model. How does the train-test gap change?
3. **Add augmentation to the `Dataset`** — give `MalariaDataset` a `train=True` flag and randomly flip the image in `__getitem__` when it’s set. Does the train-test gap shrink?
4. **Replace a MaxPool with a stride-2 conv** (`Conv2d(..., stride=2)`, remove that block’s pool). Same output size — does it train differently?

In [ ]:
# ===== YOUR EXPERIMENTS HERE =====



---
## Summary

- **The building blocks:** **pooling** shrinks a feature map while keeping the strongest signal; **stride** is the filter’s step size (stride 2 halves the output); **padding** adds a border so the size is preserved and edges are used fairly.
- **The blueprint:** **[Conv → ReLU → Pool] × N → Flatten → Dense → output** — a feature extractor then a classifier.
- **Track the shape** through each block (64→32→16→8) so the flatten size is right — the #1 skill for building CNNs without errors.
- **Overfitting** shows up as a growing train–test gap; **dropout** and **data augmentation** are the standard defenses.
- You built a **malaria detector at ~95% accuracy** from raw cell images — a real, useful computer-vision task.

> In medicine the confusion matrix matters as much as the accuracy: a missed infection (false negative) is the costly mistake. Always judge a model by *which* errors it makes.

**Tomorrow:** a short Keras version of a CNN for cross-framework fluency, another dataset, and an assessed lab where you build a CNN on your own.